# Financial Time-Series Analysis & Prediction

## A Comprehensive Guide to Stock Market Analysis with Python

---

> **Disclaimer**: This notebook is for **educational and portfolio demonstration purposes only**. Nothing here constitutes financial advice. Past performance does not guarantee future results. Markets are complex adaptive systems -- approach any "prediction" with deep skepticism.

---

### What We Will Build

This notebook walks through a **complete financial data science pipeline**, covering:

| Section | Description |
|---------|-------------|
| **1. Data Acquisition** | Fetching 5 years of FAANG + SPY daily data via `yfinance` (with synthetic fallback) |
| **2. Exploratory Analysis** | Normalized prices, return distributions, volatility, correlations, drawdowns |
| **3. Technical Indicators** | Hand-coded SMA, EMA, RSI, MACD, Bollinger Bands, ATR, OBV |
| **4. Feature Engineering** | Lag features, rolling stats, calendar effects -- all without look-ahead bias |
| **5. Prediction Models** | Linear Regression, Random Forest, XGBoost, LightGBM, LSTM with walk-forward validation |
| **6. Strategy Backtesting** | SMA crossover, ML-based strategy vs. buy-and-hold with performance metrics |
| **7. Risk Analysis** | VaR, Expected Shortfall, Monte Carlo simulation, mean-variance portfolio optimization |

### Why This Matters

Financial time-series analysis sits at the intersection of statistics, machine learning, and domain expertise. Unlike typical ML problems:

- **Temporal ordering is sacred** -- you cannot randomly shuffle train/test splits.
- **Non-stationarity** -- the data-generating process itself changes over time.
- **Signal-to-noise ratio is extremely low** -- most "patterns" are noise.
- **Survivorship bias** -- we only analyze stocks that still exist today.
- **Transaction costs and slippage** -- a model profitable on paper may lose money in practice.

We address each of these throughout the notebook.

---
## 1. Setup & Data Acquisition
---

In [ ]:
# ============================================================
# 1.1  Install & Import Dependencies
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from datetime import datetime, timedelta
import itertools, os

# Interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'

# ML
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed -- will skip XGBoost model.')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('lightgbm not installed -- will skip LightGBM model.')

# Deep learning -- try torch first, then keras
HAS_TORCH = False
HAS_KERAS = False
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
except ImportError:
    try:
        from tensorflow import keras
        from tensorflow.keras import layers
        HAS_KERAS = True
    except ImportError:
        print('Neither PyTorch nor TensorFlow found -- will skip LSTM model.')

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('All imports successful.')

In [ ]:
# ============================================================
# 1.2  Fetch Data  (yfinance with synthetic fallback)
# ============================================================

TICKERS = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL', 'SPY']
END_DATE = datetime.today()
START_DATE = END_DATE - timedelta(days=5 * 365)

def fetch_real_data(tickers, start, end):
    """Attempt to fetch real market data via yfinance."""
    import yfinance as yf
    data = yf.download(tickers, start=start, end=end, auto_adjust=True)
    return data

def generate_synthetic_data(tickers, start, end):
    """
    Fallback: generate realistic synthetic stock data using
    Geometric Brownian Motion with parameters calibrated to
    approximate real stock behavior.
    """
    print('Generating synthetic data (yfinance unavailable)...')
    dates = pd.bdate_range(start=start, end=end)  # business days
    n = len(dates)
    np.random.seed(RANDOM_SEED)

    # Approximate annualized parameters per ticker
    params = {
        'META':  {'mu': 0.20, 'sigma': 0.38, 'S0': 170},
        'AAPL':  {'mu': 0.22, 'sigma': 0.28, 'S0': 150},
        'AMZN':  {'mu': 0.18, 'sigma': 0.32, 'S0': 100},
        'NFLX':  {'mu': 0.25, 'sigma': 0.42, 'S0': 350},
        'GOOGL': {'mu': 0.19, 'sigma': 0.27, 'S0': 95},
        'SPY':   {'mu': 0.10, 'sigma': 0.16, 'S0': 400},
    }

    close_dict = {}
    volume_dict = {}
    high_dict = {}
    low_dict = {}
    open_dict = {}

    dt = 1 / 252  # one trading day
    for t in tickers:
        p = params.get(t, {'mu': 0.12, 'sigma': 0.25, 'S0': 100})
        # GBM path
        log_returns = (p['mu'] - 0.5 * p['sigma']**2) * dt + \
                       p['sigma'] * np.sqrt(dt) * np.random.randn(n)
        log_returns[0] = 0
        prices = p['S0'] * np.exp(np.cumsum(log_returns))

        # Intraday range
        daily_vol = p['sigma'] * np.sqrt(dt)
        high = prices * (1 + np.abs(np.random.randn(n)) * daily_vol * 0.5)
        low  = prices * (1 - np.abs(np.random.randn(n)) * daily_vol * 0.5)
        opn  = low + (high - low) * np.random.rand(n)
        vol  = np.random.lognormal(mean=17, sigma=0.5, size=n).astype(int)

        close_dict[t] = prices
        high_dict[t] = high
        low_dict[t] = low
        open_dict[t] = opn
        volume_dict[t] = vol

    close_df = pd.DataFrame(close_dict, index=dates)
    high_df  = pd.DataFrame(high_dict, index=dates)
    low_df   = pd.DataFrame(low_dict, index=dates)
    open_df  = pd.DataFrame(open_dict, index=dates)
    vol_df   = pd.DataFrame(volume_dict, index=dates)

    # Build multi-level columns like yfinance returns
    data = pd.concat({
        'Close': close_df,
        'High': high_df,
        'Low': low_df,
        'Open': open_df,
        'Volume': vol_df,
    }, axis=1)
    return data

# --- Attempt real data, fall back to synthetic ---
try:
    raw_data = fetch_real_data(TICKERS, START_DATE, END_DATE)
    DATA_SOURCE = 'yfinance (real market data)'
except Exception as e:
    print(f'yfinance failed: {e}')
    raw_data = generate_synthetic_data(TICKERS, START_DATE, END_DATE)
    DATA_SOURCE = 'synthetic (GBM simulation)'

# --- Flatten multi-level columns for easier access ---
if isinstance(raw_data.columns, pd.MultiIndex):
    close_prices = raw_data['Close'][TICKERS].copy()
    volumes = raw_data['Volume'][TICKERS].copy()
    high_prices = raw_data['High'][TICKERS].copy()
    low_prices = raw_data['Low'][TICKERS].copy()
    open_prices = raw_data['Open'][TICKERS].copy()
else:
    close_prices = raw_data[TICKERS].copy()
    volumes = None
    high_prices = None
    low_prices = None
    open_prices = None

close_prices.dropna(inplace=True)

# Daily simple returns
returns = close_prices.pct_change().dropna()
# Log returns (additive over time)
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

print(f'Data source : {DATA_SOURCE}')
print(f'Date range  : {close_prices.index[0].date()} to {close_prices.index[-1].date()}')
print(f'Trading days: {len(close_prices)}')
print(f'Tickers     : {list(close_prices.columns)}')
close_prices.tail()

---
## 2. Exploratory Data Analysis
---

### 2.1 Normalized Price History

To compare stocks with vastly different price levels, we normalize each to start at 100. This lets us directly compare **cumulative returns** across all assets.

In [ ]:
# ============================================================
# 2.1  Normalized Price History (Plotly interactive)
# ============================================================
normalized = (close_prices / close_prices.iloc[0]) * 100

fig = go.Figure()
colors = px.colors.qualitative.Set1
for i, col in enumerate(normalized.columns):
    fig.add_trace(go.Scatter(
        x=normalized.index, y=normalized[col],
        name=col, mode='lines',
        line=dict(width=2, color=colors[i % len(colors)]),
    ))

fig.update_layout(
    title='Normalized Price History (Base = 100)',
    xaxis_title='Date', yaxis_title='Normalized Price',
    height=550, legend=dict(orientation='h', yanchor='bottom', y=1.02),
    hovermode='x unified',
)
fig.show()

### 2.2 Daily Returns Distributions

Stock returns are famously **not** normally distributed. They exhibit:
- **Fat tails** (extreme events more common than a Gaussian predicts)
- **Slight negative skew** (crashes are sharper than rallies)
- **Excess kurtosis** (leptokurtic distributions)

We verify this below with histograms and QQ plots.

In [ ]:
# ============================================================
# 2.2  Return Distributions: Histograms + QQ Plots
# ============================================================
faang = [t for t in TICKERS if t != 'SPY']

fig, axes = plt.subplots(2, len(faang), figsize=(20, 8), constrained_layout=True)

for i, ticker in enumerate(faang):
    r = returns[ticker].dropna()

    # Histogram with fitted normal
    ax = axes[0, i]
    ax.hist(r, bins=80, density=True, alpha=0.65, color=colors[i], edgecolor='white', linewidth=0.3)
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 200)
    mu, sigma = r.mean(), r.std()
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'k--', lw=1.5, label='Normal fit')
    ax.set_title(f'{ticker}  (skew={r.skew():.2f}, kurt={r.kurtosis():.2f})')
    ax.legend(fontsize=8)
    if i == 0:
        ax.set_ylabel('Density')

    # QQ plot
    ax2 = axes[1, i]
    stats.probplot(r, dist='norm', plot=ax2)
    ax2.get_lines()[0].set_markerfacecolor(colors[i])
    ax2.get_lines()[0].set_markersize(2)
    ax2.set_title(f'{ticker} QQ Plot')

fig.suptitle('Daily Return Distributions vs. Normal', fontsize=16, y=1.02)
plt.show()

# Summary statistics
summary = returns.describe().T
summary['skew'] = returns.skew()
summary['kurtosis'] = returns.kurtosis()
summary['annualized_return'] = returns.mean() * 252
summary['annualized_vol'] = returns.std() * np.sqrt(252)
summary[['mean', 'std', 'skew', 'kurtosis', 'annualized_return', 'annualized_vol']].round(4)

### 2.3 Rolling Volatility

Volatility is not constant -- it **clusters**. Periods of high volatility beget more high volatility (heteroskedasticity). This is a fundamental stylized fact of financial returns.

In [ ]:
# ============================================================
# 2.3  Rolling Volatility (annualized, 21-day window)
# ============================================================
rolling_vol = returns.rolling(window=21).std() * np.sqrt(252)

fig = go.Figure()
for i, col in enumerate(rolling_vol.columns):
    fig.add_trace(go.Scatter(
        x=rolling_vol.index, y=rolling_vol[col],
        name=col, mode='lines',
        line=dict(width=1.5, color=colors[i % len(colors)]),
    ))

fig.update_layout(
    title='21-Day Rolling Annualized Volatility',
    xaxis_title='Date', yaxis_title='Annualized Volatility',
    height=450, legend=dict(orientation='h', yanchor='bottom', y=1.02),
    hovermode='x unified',
)
fig.show()

### 2.4 Correlation Analysis

Correlations between returns matter for diversification. High correlations mean less diversification benefit.

In [ ]:
# ============================================================
# 2.4  Correlation Matrix
# ============================================================
corr = returns.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns, y=corr.index,
    colorscale='RdBu_r', zmin=-1, zmax=1,
    text=np.round(corr.values, 2), texttemplate='%{text}',
    textfont=dict(size=13),
))
fig.update_layout(
    title='Return Correlation Matrix',
    height=500, width=600,
)
fig.show()

# Rolling correlation of each stock with SPY
fig2, ax = plt.subplots(figsize=(14, 5))
for i, t in enumerate(faang):
    rolling_corr = returns[t].rolling(63).corr(returns['SPY'])
    ax.plot(rolling_corr.index, rolling_corr, label=t, linewidth=1.2)
ax.set_title('63-Day Rolling Correlation with SPY')
ax.set_ylabel('Correlation')
ax.legend()
ax.axhline(y=0, color='grey', linestyle='--', linewidth=0.7)
plt.tight_layout()
plt.show()

### 2.5 Drawdown Analysis

**Drawdown** measures the decline from a historical peak. It captures the pain an investor experiences holding through a downturn. Maximum drawdown is one of the most important risk metrics.

In [ ]:
# ============================================================
# 2.5  Drawdown Analysis
# ============================================================
def compute_drawdown(price_series):
    """Compute drawdown series from price series."""
    cummax = price_series.cummax()
    drawdown = (price_series - cummax) / cummax
    return drawdown

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=['Normalized Prices', 'Drawdown from Peak'],
    vertical_spacing=0.08,
)

for i, col in enumerate(TICKERS):
    # Top: normalized prices
    fig.add_trace(go.Scatter(
        x=normalized.index, y=normalized[col],
        name=col, legendgroup=col,
        line=dict(width=1.5, color=colors[i % len(colors)]),
    ), row=1, col=1)

    # Bottom: drawdowns
    dd = compute_drawdown(close_prices[col])
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd,
        name=col, legendgroup=col, showlegend=False,
        line=dict(width=1.2, color=colors[i % len(colors)]),
        fill='tozeroy', fillcolor=f'rgba({i*40},{100},{200-i*30},0.08)',
    ), row=2, col=1)

fig.update_layout(height=650, hovermode='x unified',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.update_yaxes(title_text='Normalized Price', row=1)
fig.update_yaxes(title_text='Drawdown', tickformat='.0%', row=2)
fig.show()

# Max drawdown table
max_dd = {t: compute_drawdown(close_prices[t]).min() for t in TICKERS}
dd_df = pd.DataFrame.from_dict(max_dd, orient='index', columns=['Max Drawdown'])
dd_df['Max Drawdown'] = dd_df['Max Drawdown'].map('{:.1%}'.format)
dd_df

### 2.6 Volume Analysis

In [ ]:
# ============================================================
# 2.6  Volume Analysis
# ============================================================
if volumes is not None:
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.4],
        subplot_titles=['AAPL Close Price', 'AAPL Volume (20-day MA)'],
        vertical_spacing=0.08,
    )
    fig.add_trace(go.Scatter(
        x=close_prices.index, y=close_prices['AAPL'],
        name='Close', line=dict(color='steelblue', width=1.5),
    ), row=1, col=1)

    vol_ma = volumes['AAPL'].rolling(20).mean()
    fig.add_trace(go.Bar(
        x=volumes.index, y=volumes['AAPL'],
        name='Daily Volume', marker_color='lightgrey', opacity=0.5,
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=vol_ma.index, y=vol_ma,
        name='20-day MA', line=dict(color='darkorange', width=2),
    ), row=2, col=1)

    fig.update_layout(height=550, hovermode='x unified',
                      legend=dict(orientation='h', yanchor='bottom', y=1.02))
    fig.show()
else:
    print('Volume data not available.')

---
## 3. Technical Indicators (Implemented from Scratch)

We implement each indicator from first principles -- no `ta` or `talib` library -- to demonstrate understanding of the underlying mathematics.

---

In [ ]:
# ============================================================
# 3.0  Technical Indicator Implementations
# ============================================================

def sma(series, window):
    """Simple Moving Average: unweighted mean of the last `window` values."""
    return series.rolling(window=window).mean()


def ema(series, span):
    """
    Exponential Moving Average.
    Uses the standard smoothing factor alpha = 2 / (span + 1).
    More recent data points receive exponentially higher weight.
    """
    return series.ewm(span=span, adjust=False).mean()


def rsi(series, period=14):
    """
    Relative Strength Index (Wilder's smoothing).
    
    RSI = 100 - 100 / (1 + RS)
    where RS = avg_gain / avg_loss
    
    Uses exponential smoothing with alpha=1/period (Wilder's method).
    Values above 70 suggest overbought; below 30 suggest oversold.
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing (equivalent to EMA with alpha=1/period)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def macd(series, fast=12, slow=26, signal=9):
    """
    Moving Average Convergence Divergence.
    
    MACD Line   = EMA(fast) - EMA(slow)
    Signal Line = EMA(MACD Line, signal)
    Histogram   = MACD Line - Signal Line
    
    Bullish signal when MACD crosses above signal line.
    Returns: (macd_line, signal_line, histogram)
    """
    ema_fast = ema(series, span=fast)
    ema_slow = ema(series, span=slow)
    macd_line = ema_fast - ema_slow
    signal_line = ema(macd_line, span=signal)
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram


def bollinger_bands(series, window=20, num_std=2):
    """
    Bollinger Bands.
    
    Middle = SMA(window)
    Upper  = Middle + num_std * rolling_std
    Lower  = Middle - num_std * rolling_std
    
    Bandwidth measures volatility; price touching bands may signal
    mean-reversion or breakout depending on context.
    Returns: (upper, middle, lower, bandwidth)
    """
    middle = sma(series, window)
    std = series.rolling(window=window).std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    bandwidth = (upper - lower) / middle  # relative width
    return upper, middle, lower, bandwidth


def atr(high, low, close, period=14):
    """
    Average True Range -- measures market volatility.
    
    True Range = max(High-Low, |High-PrevClose|, |Low-PrevClose|)
    ATR = Wilder's smoothed average of True Range over `period` days.
    
    Higher ATR = higher volatility. Used for position sizing and stop-losses.
    """
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return true_range.ewm(alpha=1/period, min_periods=period, adjust=False).mean()


def obv(close, volume):
    """
    On-Balance Volume -- relates volume to price direction.
    
    If close > prev_close: add volume
    If close < prev_close: subtract volume
    
    Rising OBV confirms uptrend; divergence from price may signal reversal.
    """
    direction = np.sign(close.diff())
    direction.iloc[0] = 0
    return (volume * direction).cumsum()


print('All technical indicator functions defined.')

In [ ]:
# ============================================================
# 3.1  Visualize All Indicators on AAPL
# ============================================================
ticker = 'AAPL'
px_aapl = close_prices[ticker]
hi_aapl = high_prices[ticker] if high_prices is not None else px_aapl * 1.005
lo_aapl = low_prices[ticker] if low_prices is not None else px_aapl * 0.995
vol_aapl = volumes[ticker] if volumes is not None else pd.Series(
    np.random.randint(1_000_000, 100_000_000, len(px_aapl)), index=px_aapl.index
)

# Compute all indicators
sma_50  = sma(px_aapl, 50)
sma_200 = sma(px_aapl, 200)
ema_20  = ema(px_aapl, 20)
rsi_14  = rsi(px_aapl, 14)
macd_line, signal_line, macd_hist = macd(px_aapl)
bb_upper, bb_middle, bb_lower, bb_bw = bollinger_bands(px_aapl)
atr_14  = atr(hi_aapl, lo_aapl, px_aapl, 14)
obv_val = obv(px_aapl, vol_aapl)

# --- Multi-panel chart ---
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True,
    row_heights=[0.35, 0.15, 0.15, 0.15, 0.15],
    subplot_titles=[
        f'{ticker} Price with Bollinger Bands & Moving Averages',
        'RSI (14)', 'MACD', 'ATR (14)', 'OBV'
    ],
    vertical_spacing=0.04,
)

# Panel 1: Price + BB + SMAs
fig.add_trace(go.Scatter(x=px_aapl.index, y=px_aapl, name='Close',
                         line=dict(color='#1f77b4', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=px_aapl.index, y=bb_upper, name='BB Upper',
                         line=dict(color='grey', width=0.8, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=px_aapl.index, y=bb_lower, name='BB Lower',
                         line=dict(color='grey', width=0.8, dash='dot'),
                         fill='tonexty', fillcolor='rgba(173,216,230,0.15)'), row=1, col=1)
fig.add_trace(go.Scatter(x=px_aapl.index, y=sma_50, name='SMA 50',
                         line=dict(color='orange', width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=px_aapl.index, y=sma_200, name='SMA 200',
                         line=dict(color='red', width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=px_aapl.index, y=ema_20, name='EMA 20',
                         line=dict(color='green', width=1, dash='dash')), row=1, col=1)

# Panel 2: RSI
fig.add_trace(go.Scatter(x=rsi_14.index, y=rsi_14, name='RSI',
                         line=dict(color='purple', width=1.2)), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', line_width=0.8, row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', line_width=0.8, row=2, col=1)

# Panel 3: MACD
hist_colors = ['green' if v >= 0 else 'red' for v in macd_hist]
fig.add_trace(go.Bar(x=macd_hist.index, y=macd_hist, name='MACD Hist',
                     marker_color=hist_colors, opacity=0.5), row=3, col=1)
fig.add_trace(go.Scatter(x=macd_line.index, y=macd_line, name='MACD',
                         line=dict(color='blue', width=1.2)), row=3, col=1)
fig.add_trace(go.Scatter(x=signal_line.index, y=signal_line, name='Signal',
                         line=dict(color='red', width=1, dash='dash')), row=3, col=1)

# Panel 4: ATR
fig.add_trace(go.Scatter(x=atr_14.index, y=atr_14, name='ATR',
                         line=dict(color='darkorange', width=1.2),
                         fill='tozeroy', fillcolor='rgba(255,165,0,0.1)'), row=4, col=1)

# Panel 5: OBV
fig.add_trace(go.Scatter(x=obv_val.index, y=obv_val, name='OBV',
                         line=dict(color='teal', width=1.2)), row=5, col=1)

fig.update_layout(height=1100, showlegend=True,
                  legend=dict(orientation='h', yanchor='bottom', y=1.01, font=dict(size=9)),
                  hovermode='x unified')
fig.update_yaxes(title_text='Price ($)', row=1)
fig.update_yaxes(title_text='RSI', row=2)
fig.update_yaxes(title_text='MACD', row=3)
fig.update_yaxes(title_text='ATR ($)', row=4)
fig.update_yaxes(title_text='OBV', row=5)
fig.show()

---
## 4. Feature Engineering for Prediction

We engineer features from historical data, being **extremely careful** about look-ahead bias:

- All features use **only past data** (no future information leaks).
- Rolling statistics use `min_periods` equal to the window to avoid partial windows.
- We predict the **next-day return**, so the target at time $t$ is $r_{t+1}$.

---

In [ ]:
# ============================================================
# 4.0  Feature Engineering (AAPL as demonstration)
# ============================================================

def build_features(close, high, low, volume, ticker_name='AAPL'):
    """
    Build a feature matrix for predicting the next-day return.
    
    CRITICAL: Every feature at index t uses only data from time <= t.
    The target is the return from t to t+1.
    """
    df = pd.DataFrame(index=close.index)
    ret = close.pct_change()

    # --- Lag features: returns at t-0, t-1, ..., t-5 ---
    # These capture short-term momentum and mean-reversion patterns
    for lag in range(0, 6):
        df[f'ret_lag_{lag}'] = ret.shift(lag)

    # --- Rolling statistics at various windows ---
    # Short windows (5d, 10d) capture recent trends
    # Longer windows (21d ~1mo, 63d ~1qtr) capture regime
    for w in [5, 10, 21, 63]:
        df[f'ret_mean_{w}d']  = ret.rolling(w, min_periods=w).mean()
        df[f'ret_std_{w}d']   = ret.rolling(w, min_periods=w).std()
        df[f'ret_min_{w}d']   = ret.rolling(w, min_periods=w).min()
        df[f'ret_max_{w}d']   = ret.rolling(w, min_periods=w).max()
        df[f'price_vs_sma_{w}'] = close / sma(close, w) - 1  # % above/below SMA

    # --- Technical indicators ---
    df['rsi_14'] = rsi(close, 14)
    macd_l, sig_l, hist_l = macd(close)
    df['macd_line'] = macd_l
    df['macd_signal'] = sig_l
    df['macd_hist'] = hist_l
    bb_u, bb_m, bb_l, bb_bw = bollinger_bands(close)
    df['bb_bandwidth'] = bb_bw
    df['bb_pct'] = (close - bb_l) / (bb_u - bb_l)  # Position within bands [0,1]
    df['atr_14'] = atr(high, low, close, 14)
    df['atr_pct'] = df['atr_14'] / close  # ATR as % of price (normalized)

    # --- Volume features ---
    df['vol_ratio_20'] = volume / volume.rolling(20, min_periods=20).mean()
    df['obv_slope'] = obv(close, volume).diff(5)  # OBV momentum (5-day change)

    # --- Calendar effects ---
    # The "Monday effect" and "January effect" are well-documented anomalies
    df['day_of_week'] = close.index.dayofweek
    df['month'] = close.index.month
    df['is_month_end'] = close.index.is_month_end.astype(int)
    df['is_quarter_end'] = close.index.is_quarter_end.astype(int)

    # --- Target: next-day return (shifted forward) ---
    # This is what we are trying to predict
    df['target'] = ret.shift(-1)  # return from t to t+1

    # Drop rows with NaN (from rolling windows & shifts)
    df.dropna(inplace=True)
    return df


# Build feature set for AAPL
feat_df = build_features(px_aapl, hi_aapl, lo_aapl, vol_aapl)
feature_cols = [c for c in feat_df.columns if c != 'target']

print(f'Feature matrix shape: {feat_df.shape}')
print(f'Date range: {feat_df.index[0].date()} to {feat_df.index[-1].date()}')
print(f'\nFeatures ({len(feature_cols)} total):')
for i, c in enumerate(feature_cols):
    print(f'  {i+1:2d}. {c}')

In [ ]:
# ============================================================
# 4.1  Feature Importance Preview (correlation with target)
# ============================================================
corr_with_target = feat_df[feature_cols].corrwith(feat_df['target']).sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
corr_with_target.plot(kind='barh', ax=ax, color=[
    'steelblue' if v > 0 else 'salmon' for v in corr_with_target
])
ax.set_title('Feature Correlation with Next-Day Return')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\nNote: Low correlations are EXPECTED for stock returns.')
print('The signal-to-noise ratio in financial data is notoriously low.')
print('Even small consistent edges (|r| > 0.02) can be economically meaningful.')

---
## 5. Prediction Models

### Why Walk-Forward Validation?

In time-series, **random train/test splits are invalid** because they allow future data to leak into training. Instead, we use **walk-forward (expanding window) validation**:

1. Train on data from the start up to time $t$.
2. Predict the next period $t+1$.
3. Move $t$ forward and repeat.

This mimics real-world deployment where you only have historical data available when making predictions.

```
Time:  [====== TRAIN ======][TEST]     <- Period 1
       [======== TRAIN ========][TEST] <- Period 2
       [========== TRAIN ==========][TEST] <- Period 3
```

---

In [ ]:
# ============================================================
# 5.0  Walk-Forward Validation Framework
# ============================================================

X = feat_df[feature_cols].values
y = feat_df['target'].values
dates_idx = feat_df.index

# We use the last 252 trading days (~1 year) as the out-of-sample test period.
# The model is retrained every 21 days (monthly) on all available history.
TEST_SIZE = min(252, len(X) // 3)  # ensure enough training data
RETRAIN_FREQ = 21

train_end = len(X) - TEST_SIZE
test_indices = list(range(train_end, len(X)))

print(f'Total samples     : {len(X)}')
print(f'Initial train size: {train_end}')
print(f'Test size          : {TEST_SIZE}')
print(f'Test period        : {dates_idx[train_end].date()} to {dates_idx[-1].date()}')
print(f'Retrain frequency  : every {RETRAIN_FREQ} days')


def walk_forward_predict(model_class, model_params, X, y, train_end, test_indices,
                         retrain_freq=21, scale=True):
    """
    Walk-forward prediction with periodic retraining.
    
    - Expands training window as we move forward in time
    - Retrains the model every `retrain_freq` steps
    - Scales features using only training data (no leakage)
    
    Returns array of predictions aligned with test_indices.
    """
    predictions = []
    scaler = StandardScaler() if scale else None
    model = None

    for i, idx in enumerate(test_indices):
        # Retrain periodically on expanding window
        if i % retrain_freq == 0 or model is None:
            X_train = X[:train_end + i]
            y_train = y[:train_end + i]
            if scale:
                X_train_s = scaler.fit_transform(X_train)
            else:
                X_train_s = X_train
            model = model_class(**model_params)
            model.fit(X_train_s, y_train)

        # Predict (using scaler fitted on training data only)
        x_test = X[idx:idx+1]
        if scale:
            x_test = scaler.transform(x_test)
        pred = model.predict(x_test)[0]
        predictions.append(pred)

    return np.array(predictions)


def compute_metrics(y_true, y_pred, label=''):
    """Compute regression and directional accuracy metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_ = mean_absolute_error(y_true, y_pred)
    # Directional accuracy: did we predict the correct sign of the return?
    dir_acc = np.mean(np.sign(y_pred) == np.sign(y_true))
    return {'Model': label, 'RMSE': rmse, 'MAE': mae_, 'Direction Accuracy': dir_acc}

In [ ]:
# ============================================================
# 5.1  Model Training & Assessment
# ============================================================

y_test = y[train_end:]
results = []
all_preds = {}

# --- 1) Linear Regression (baseline) ---
print('Training Linear Regression (baseline)...')
lr_preds = walk_forward_predict(
    LinearRegression, {}, X, y, train_end, test_indices, RETRAIN_FREQ, scale=True
)
results.append(compute_metrics(y_test, lr_preds, 'Linear Regression'))
all_preds['Linear Regression'] = lr_preds
print('  Done.')

# --- 2) Random Forest ---
print('Training Random Forest...')
rf_preds = walk_forward_predict(
    RandomForestRegressor,
    {'n_estimators': 200, 'max_depth': 6, 'min_samples_leaf': 20,
     'random_state': RANDOM_SEED, 'n_jobs': -1},
    X, y, train_end, test_indices, RETRAIN_FREQ, scale=False
)
results.append(compute_metrics(y_test, rf_preds, 'Random Forest'))
all_preds['Random Forest'] = rf_preds
print('  Done.')

# --- 3) XGBoost ---
if HAS_XGB:
    print('Training XGBoost...')
    xgb_preds = walk_forward_predict(
        xgb.XGBRegressor,
        {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05,
         'subsample': 0.8, 'colsample_bytree': 0.8,
         'random_state': RANDOM_SEED, 'verbosity': 0},
        X, y, train_end, test_indices, RETRAIN_FREQ, scale=False
    )
    results.append(compute_metrics(y_test, xgb_preds, 'XGBoost'))
    all_preds['XGBoost'] = xgb_preds
    print('  Done.')

# --- 4) LightGBM ---
if HAS_LGB:
    print('Training LightGBM...')
    lgb_preds = walk_forward_predict(
        lgb.LGBMRegressor,
        {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05,
         'subsample': 0.8, 'colsample_bytree': 0.8,
         'random_state': RANDOM_SEED, 'verbosity': -1},
        X, y, train_end, test_indices, RETRAIN_FREQ, scale=False
    )
    results.append(compute_metrics(y_test, lgb_preds, 'LightGBM'))
    all_preds['LightGBM'] = lgb_preds
    print('  Done.')

# Display results so far
results_df = pd.DataFrame(results).set_index('Model')
results_df.style.format({
    'RMSE': '{:.6f}', 'MAE': '{:.6f}', 'Direction Accuracy': '{:.2%}'
})

In [ ]:
# ============================================================
# 5.2  LSTM Model (PyTorch or Keras)
# ============================================================

SEQUENCE_LENGTH = 20  # look back 20 trading days

if HAS_TORCH:
    print('Building LSTM with PyTorch...')

    class StockDataset(Dataset):
        """Custom PyTorch dataset for sequential stock data."""
        def __init__(self, X, y, seq_len):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32)
            self.seq_len = seq_len

        def __len__(self):
            return len(self.X) - self.seq_len

        def __getitem__(self, idx):
            return (self.X[idx:idx+self.seq_len],
                    self.y[idx+self.seq_len])

    class LSTMModel(nn.Module):
        """
        Two-layer LSTM with dropout for regularization.
        Takes a sequence of feature vectors and outputs a single return prediction.
        """
        def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                                batch_first=True, dropout=dropout)
            self.fc = nn.Sequential(
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1),
            )

        def forward(self, x):
            lstm_out, _ = self.lstm(x)
            # Use the last time step's hidden state
            last = lstm_out[:, -1, :]
            return self.fc(last).squeeze(-1)

    # Scale features using ONLY training data
    scaler_lstm = StandardScaler()
    X_scaled_train = scaler_lstm.fit_transform(X[:train_end])
    X_all_scaled = scaler_lstm.transform(X)

    # Create training dataset and loader
    train_dataset = StockDataset(X_all_scaled[:train_end], y[:train_end], SEQUENCE_LENGTH)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    # Initialize model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_lstm = LSTMModel(input_dim=X.shape[1]).to(device)
    optimizer = torch.optim.Adam(model_lstm.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    # Training loop
    EPOCHS = 30
    for epoch in range(EPOCHS):
        model_lstm.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model_lstm(X_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            # Gradient clipping to prevent exploding gradients in RNNs
            torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.6f}')

    # Walk-forward prediction for LSTM
    model_lstm.eval()
    lstm_preds = []
    with torch.no_grad():
        for idx in test_indices:
            if idx >= SEQUENCE_LENGTH:
                seq = torch.tensor(
                    X_all_scaled[idx-SEQUENCE_LENGTH:idx],
                    dtype=torch.float32
                ).unsqueeze(0).to(device)
                pred = model_lstm(seq).item()
            else:
                pred = 0.0
            lstm_preds.append(pred)

    lstm_preds = np.array(lstm_preds)
    results.append(compute_metrics(y_test, lstm_preds, 'LSTM (PyTorch)'))
    all_preds['LSTM'] = lstm_preds
    print('  LSTM training complete.')

elif HAS_KERAS:
    print('Building LSTM with Keras...')

    scaler_lstm = StandardScaler()
    scaler_lstm.fit(X[:train_end])
    X_all_scaled = scaler_lstm.transform(X)

    def create_sequences(data, targets, seq_len):
        """Convert flat arrays into overlapping sequences for LSTM input."""
        Xs, ys = [], []
        for i in range(len(data) - seq_len):
            Xs.append(data[i:i+seq_len])
            ys.append(targets[i+seq_len])
        return np.array(Xs), np.array(ys)

    X_train_seq, y_train_seq = create_sequences(
        X_all_scaled[:train_end], y[:train_end], SEQUENCE_LENGTH
    )

    model_lstm = keras.Sequential([
        layers.LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, X.shape[1])),
        layers.Dropout(0.2),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])
    model_lstm.compile(optimizer='adam', loss='mse')
    model_lstm.fit(X_train_seq, y_train_seq, epochs=30, batch_size=64, verbose=0)
    print('  Keras LSTM trained (30 epochs).')

    # Walk-forward prediction
    lstm_preds = []
    for idx in test_indices:
        if idx >= SEQUENCE_LENGTH:
            seq = X_all_scaled[idx-SEQUENCE_LENGTH:idx].reshape(1, SEQUENCE_LENGTH, -1)
            pred = model_lstm.predict(seq, verbose=0)[0, 0]
        else:
            pred = 0.0
        lstm_preds.append(pred)

    lstm_preds = np.array(lstm_preds)
    results.append(compute_metrics(y_test, lstm_preds, 'LSTM (Keras)'))
    all_preds['LSTM'] = lstm_preds
    print('  LSTM prediction complete.')

else:
    print('Skipping LSTM (no deep learning framework available).')

# Updated results table
results_df = pd.DataFrame(results).set_index('Model')
results_df.style.format({
    'RMSE': '{:.6f}', 'MAE': '{:.6f}', 'Direction Accuracy': '{:.2%}'
})

In [ ]:
# ============================================================
# 5.3  Prediction Visualization
# ============================================================
test_dates = dates_idx[train_end:]

# Pick the best model by directional accuracy
best_model = results_df['Direction Accuracy'].idxmax()
best_preds = all_preds.get(best_model, list(all_preds.values())[0])
best_name = best_model

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.5, 0.5],
    subplot_titles=[
        f'Actual vs. Predicted Daily Returns ({best_name})',
        'Cumulative Returns: Model Strategies vs. Buy & Hold'
    ],
    vertical_spacing=0.1,
)

# Top panel: actual vs predicted returns
fig.add_trace(go.Scatter(
    x=test_dates, y=y_test,
    name='Actual Returns', line=dict(color='steelblue', width=0.8), opacity=0.6,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=test_dates, y=best_preds,
    name=f'{best_name} Predictions', line=dict(color='red', width=0.8), opacity=0.7,
), row=1, col=1)

# Bottom panel: cumulative returns comparison
cum_actual = (1 + pd.Series(y_test, index=test_dates)).cumprod()
for name, preds in all_preds.items():
    # Strategy: go long when prediction > 0, flat otherwise (no shorting)
    positions = np.where(preds > 0, 1, 0)
    strategy_returns = positions * y_test
    cum_strat = (1 + pd.Series(strategy_returns, index=test_dates)).cumprod()
    fig.add_trace(go.Scatter(
        x=test_dates, y=cum_strat, name=f'{name} Strategy',
        line=dict(width=1.5),
    ), row=2, col=1)

fig.add_trace(go.Scatter(
    x=test_dates, y=cum_actual, name='Buy & Hold',
    line=dict(color='black', width=2, dash='dash'),
), row=2, col=1)

fig.update_layout(height=700, hovermode='x unified',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, font=dict(size=9)))
fig.update_yaxes(title_text='Daily Return', row=1)
fig.update_yaxes(title_text='Cumulative Return (1 = start)', row=2)
fig.show()

---
## 6. Strategy Backtesting

We implement and compare two strategies against a buy-and-hold baseline:

1. **SMA Crossover** (classic technical strategy): Go long when SMA(50) > SMA(200), flat otherwise.
2. **ML-Based Strategy**: Go long when the best model predicts a positive return.

We measure performance using standard quantitative metrics:
- **Sharpe Ratio** -- risk-adjusted return (higher is better)
- **Max Drawdown** -- worst peak-to-trough decline (closer to 0 is better)
- **Win Rate** -- fraction of profitable trading days
- **Profit Factor** -- gross profits divided by gross losses
- **Calmar Ratio** -- annualized return divided by max drawdown

> **Important**: These backtests do NOT include transaction costs, slippage, or market impact. Real-world performance would be worse.

---

In [ ]:
# ============================================================
# 6.0  Backtesting Framework
# ============================================================

def backtest_metrics(returns_series, risk_free_rate=0.04):
    """
    Compute standard strategy performance metrics.
    Assumes daily returns and 252 trading days per year.
    """
    # Annualized return (geometric)
    total_return = (1 + returns_series).prod() - 1
    n_years = len(returns_series) / 252
    ann_return = (1 + total_return) ** (1 / max(n_years, 0.01)) - 1

    # Annualized volatility
    ann_vol = returns_series.std() * np.sqrt(252)

    # Sharpe ratio (excess return per unit of risk)
    sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else 0

    # Max drawdown
    cum = (1 + returns_series).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()

    # Win rate (fraction of days with positive returns)
    trading_days = returns_series[returns_series != 0]
    wins = (trading_days > 0).sum()
    losses = (trading_days < 0).sum()
    win_rate = wins / max(wins + losses, 1)

    # Profit factor (total gains / total losses)
    gross_profit = returns_series[returns_series > 0].sum()
    gross_loss = -returns_series[returns_series < 0].sum()
    profit_factor = gross_profit / max(gross_loss, 1e-10)

    # Calmar ratio (return / max drawdown)
    calmar = ann_return / abs(max_dd) if max_dd != 0 else 0

    return {
        'Total Return': total_return,
        'Ann. Return': ann_return,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Win Rate': win_rate,
        'Profit Factor': profit_factor,
    }


# --- Test period data ---
test_close = px_aapl.iloc[train_end:]
test_returns_raw = returns['AAPL'].iloc[train_end:].values

# Align lengths
min_len = min(len(test_returns_raw), len(y_test))
test_ret = test_returns_raw[:min_len]
test_dates_bt = test_dates[:min_len]

# --- Strategy 1: SMA Crossover (50/200) ---
# Classic "Golden Cross / Death Cross" strategy
sma50_test = sma(px_aapl, 50).iloc[train_end:].values[:min_len]
sma200_test = sma(px_aapl, 200).iloc[train_end:].values[:min_len]
sma_signal = np.where(sma50_test > sma200_test, 1, 0)  # 1 = long, 0 = flat
sma_returns = sma_signal * test_ret

# --- Strategy 2: ML-Based (best model) ---
ml_signal = np.where(best_preds[:min_len] > 0, 1, 0)
ml_returns = ml_signal * test_ret

# --- Benchmark: Buy & Hold ---
bh_returns = test_ret

# --- Compute metrics for all strategies ---
strategies = {
    'Buy & Hold': pd.Series(bh_returns, index=test_dates_bt),
    'SMA Crossover (50/200)': pd.Series(sma_returns, index=test_dates_bt),
    f'ML ({best_name})': pd.Series(ml_returns, index=test_dates_bt),
}

metrics_list = []
for name, ret_series in strategies.items():
    m = backtest_metrics(ret_series)
    m['Strategy'] = name
    metrics_list.append(m)

bt_results = pd.DataFrame(metrics_list).set_index('Strategy')
bt_results.style.format({
    'Total Return': '{:.2%}', 'Ann. Return': '{:.2%}',
    'Ann. Volatility': '{:.2%}', 'Sharpe Ratio': '{:.2f}',
    'Max Drawdown': '{:.2%}', 'Calmar Ratio': '{:.2f}',
    'Win Rate': '{:.2%}', 'Profit Factor': '{:.2f}',
})

In [ ]:
# ============================================================
# 6.1  Equity Curves & Drawdown Comparison
# ============================================================

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Equity Curves (Starting Capital = $1)', 'Strategy Drawdowns'],
    vertical_spacing=0.08,
)

strategy_colors = {
    'Buy & Hold': 'black',
    'SMA Crossover (50/200)': 'darkorange',
    f'ML ({best_name})': 'steelblue',
}

for name, ret_series in strategies.items():
    equity = (1 + ret_series).cumprod()
    color = strategy_colors.get(name, 'grey')
    is_benchmark = 'Hold' in name

    # Equity curve
    fig.add_trace(go.Scatter(
        x=equity.index, y=equity,
        name=name, mode='lines',
        line=dict(width=2.5 if is_benchmark else 2,
                  dash='dash' if is_benchmark else 'solid',
                  color=color),
    ), row=1, col=1)

    # Drawdown
    dd = equity / equity.cummax() - 1
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd,
        name=name, showlegend=False, mode='lines',
        line=dict(width=1.2, color=color),
        fill='tozeroy', opacity=0.3,
    ), row=2, col=1)

fig.update_layout(height=600, hovermode='x unified',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.update_yaxes(title_text='Portfolio Value ($)', row=1)
fig.update_yaxes(title_text='Drawdown', tickformat='.0%', row=2)
fig.show()

---
## 7. Risk Analysis

This section covers key risk measures used by quantitative analysts and portfolio risk managers.

---

### 7.1 Value at Risk (VaR) & Expected Shortfall

- **VaR at confidence level alpha**: The loss that is exceeded with probability (1 - alpha). For example, 95% VaR = 2.1% means there is a 5% chance of losing more than 2.1% in a day.
- **Expected Shortfall (CVaR)**: The expected loss *given* that we are in the worst (1-alpha) tail. It answers: "When things go bad, *how* bad on average?"

We compute both using:
1. **Historical method** -- empirical quantile of past returns (makes no distributional assumption)
2. **Parametric method** -- assumes returns follow a normal distribution (underestimates tail risk)

In [ ]:
# ============================================================
# 7.1  VaR and Expected Shortfall
# ============================================================

confidence_levels = [0.95, 0.99]
var_results = []

for ticker in TICKERS:
    r = returns[ticker].dropna()
    mu_r, sigma_r = r.mean(), r.std()

    for alpha in confidence_levels:
        # Historical VaR (negative quantile of returns)
        hist_var = -np.percentile(r, (1 - alpha) * 100)
        # Historical ES: average of losses beyond VaR
        tail = r[r <= -hist_var]
        hist_es = -tail.mean() if len(tail) > 0 else hist_var

        # Parametric VaR (normal distribution assumption)
        z = stats.norm.ppf(1 - alpha)
        param_var = -(mu_r + z * sigma_r)
        # Parametric ES (closed-form for normal)
        param_es = -mu_r + sigma_r * stats.norm.pdf(z) / (1 - alpha)

        var_results.append({
            'Ticker': ticker,
            'Confidence': f'{alpha:.0%}',
            'Hist VaR': hist_var,
            'Hist ES': hist_es,
            'Param VaR': param_var,
            'Param ES': param_es,
        })

var_df = pd.DataFrame(var_results)
var_df.set_index(['Ticker', 'Confidence'], inplace=True)
var_df.style.format('{:.4%}')

In [ ]:
# --- VaR visualization for AAPL ---
r_aapl = returns['AAPL'].dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(r_aapl, bins=100, density=True, alpha=0.6, color='steelblue',
        edgecolor='white', label='Daily returns')

for alpha, color in [(0.95, 'orange'), (0.99, 'red')]:
    var_val = -np.percentile(r_aapl, (1 - alpha) * 100)
    ax.axvline(-var_val, color=color, linestyle='--', linewidth=2,
               label=f'VaR {alpha:.0%} = {var_val:.2%}')
    tail = r_aapl[r_aapl <= np.percentile(r_aapl, (1 - alpha) * 100)]
    es_val = -tail.mean() if len(tail) > 0 else var_val
    ax.axvline(-es_val, color=color, linestyle=':', linewidth=2,
               label=f'ES {alpha:.0%} = {es_val:.2%}')

ax.set_title('AAPL Daily Return Distribution with VaR & Expected Shortfall')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 7.2 Monte Carlo Simulation

We simulate 10,000 possible portfolio paths over the next year using **correlated returns** via Cholesky decomposition of the empirical covariance matrix. This preserves the joint dependence structure between assets.

The simulation uses **Geometric Brownian Motion**:

$$dS = \mu S \, dt + \sigma S \, dW$$

where $W$ is a correlated Wiener process.

In [ ]:
# ============================================================
# 7.2  Monte Carlo Simulation (Correlated Multi-Asset)
# ============================================================

# Equal-weighted portfolio of FAANG stocks
faang_returns = returns[faang]
n_assets = len(faang)
weights_eq = np.array([1/n_assets] * n_assets)

# Parameters from historical data
mu_vec = faang_returns.mean().values      # daily mean returns
cov_mat = faang_returns.cov().values      # daily covariance matrix

# Cholesky decomposition: L such that L @ L^T = cov_mat
# Used to generate correlated random samples
L = np.linalg.cholesky(cov_mat)

N_SIMS = 10_000
N_DAYS = 252  # simulate 1 year forward
np.random.seed(RANDOM_SEED)

# Simulate portfolio paths
portfolio_paths = np.zeros((N_SIMS, N_DAYS))
for sim in range(N_SIMS):
    # Generate correlated daily returns
    z = np.random.randn(N_DAYS, n_assets)
    correlated_returns = z @ L.T + mu_vec  # apply correlation + drift
    # Portfolio return = weighted sum of individual asset returns
    portfolio_daily_returns = correlated_returns @ weights_eq
    portfolio_paths[sim] = np.cumprod(1 + portfolio_daily_returns)

# Distribution of final portfolio values
final_values = portfolio_paths[:, -1]

# Visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Simulated Portfolio Paths (100 of {N_SIMS:,} shown)',
        'Distribution of Final Portfolio Value'
    ],
    column_widths=[0.6, 0.4],
)

# Left: Sample paths
for i in range(100):
    fig.add_trace(go.Scatter(
        x=list(range(N_DAYS)), y=portfolio_paths[i],
        mode='lines', line=dict(width=0.3, color='steelblue'),
        showlegend=False, opacity=0.15,
    ), row=1, col=1)

# Percentile bands
for pct, color, name in [(5, 'red', '5th pctile'), (50, 'black', 'Median'), (95, 'green', '95th pctile')]:
    path = np.percentile(portfolio_paths, pct, axis=0)
    fig.add_trace(go.Scatter(
        x=list(range(N_DAYS)), y=path,
        name=name, mode='lines',
        line=dict(width=2.5, color=color),
    ), row=1, col=1)

# Right: Histogram of final values
fig.add_trace(go.Histogram(
    x=final_values, nbinsx=80, marker_color='steelblue',
    opacity=0.7, showlegend=False,
), row=1, col=2)

fig.add_vline(x=1.0, line_dash='dash', line_color='red',
              annotation_text='Break-even', row=1, col=2)

fig.update_layout(height=450)
fig.update_xaxes(title_text='Trading Day', row=1, col=1)
fig.update_yaxes(title_text='Portfolio Value ($1 start)', row=1, col=1)
fig.update_xaxes(title_text='Final Value', row=1, col=2)
fig.show()

# Summary
print(f'Monte Carlo Simulation Results ({N_SIMS:,} paths, {N_DAYS} days):')
print(f'  Mean final value      : ${final_values.mean():.4f}')
print(f'  Median final value    : ${np.median(final_values):.4f}')
print(f'  Std dev of final value: ${final_values.std():.4f}')
print(f'  5th percentile        : ${np.percentile(final_values, 5):.4f}')
print(f'  95th percentile       : ${np.percentile(final_values, 95):.4f}')
print(f'  Probability of loss   : {(final_values < 1).mean():.1%}')
print(f'  Prob of >20% gain     : {(final_values > 1.2).mean():.1%}')
print(f'  1-year 95% VaR        : {1 - np.percentile(final_values, 5):.2%}')

### 7.3 Mean-Variance Portfolio Optimization

**Modern Portfolio Theory** (Markowitz, 1952): Find the portfolio weights that maximize risk-adjusted returns. Key concepts:

- **Efficient Frontier**: The set of portfolios with the highest return for each level of risk.
- **Maximum Sharpe Ratio Portfolio**: The tangency portfolio -- optimal risk-return trade-off.
- **Minimum Variance Portfolio**: The lowest-risk portfolio achievable.

We use `scipy.optimize.minimize` with constraints:
- Weights sum to 1 (fully invested)
- Weights between 0 and 1 (long-only, no shorting)

In [ ]:
# ============================================================
# 7.3  Mean-Variance Portfolio Optimization
# ============================================================

ann_returns_vec = faang_returns.mean() * 252
ann_cov_mat = faang_returns.cov() * 252
risk_free = 0.04  # 4% annual risk-free rate

def portfolio_stats(weights, ann_ret, ann_cov):
    """Compute annualized return, volatility, and Sharpe ratio for a portfolio."""
    port_ret = weights @ ann_ret
    port_vol = np.sqrt(weights @ ann_cov @ weights)
    sharpe = (port_ret - risk_free) / port_vol
    return port_ret, port_vol, sharpe

def neg_sharpe(weights, ann_ret, ann_cov):
    """Objective function: minimize negative Sharpe (i.e., maximize Sharpe)."""
    return -portfolio_stats(weights, ann_ret, ann_cov)[2]

def portfolio_vol(weights, ann_ret, ann_cov):
    """Objective function: minimize portfolio volatility."""
    return portfolio_stats(weights, ann_ret, ann_cov)[1]

# Optimization constraints and bounds
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}  # fully invested
bounds = tuple((0, 1) for _ in range(n_assets))  # long-only
w0 = weights_eq.copy()  # start from equal weights

# --- Maximum Sharpe Ratio Portfolio ---
opt_sharpe = minimize(neg_sharpe, w0,
                      args=(ann_returns_vec.values, ann_cov_mat.values),
                      method='SLSQP', bounds=bounds, constraints=constraints)
w_sharpe = opt_sharpe.x
ret_s, vol_s, sr_s = portfolio_stats(w_sharpe, ann_returns_vec.values, ann_cov_mat.values)

# --- Minimum Variance Portfolio ---
opt_minvol = minimize(portfolio_vol, w0,
                      args=(ann_returns_vec.values, ann_cov_mat.values),
                      method='SLSQP', bounds=bounds, constraints=constraints)
w_minvol = opt_minvol.x
ret_mv, vol_mv, sr_mv = portfolio_stats(w_minvol, ann_returns_vec.values, ann_cov_mat.values)

# --- Generate Efficient Frontier via Random Portfolios ---
n_portfolios = 8000
np.random.seed(RANDOM_SEED)
rand_rets, rand_vols, rand_sharpes = [], [], []

for _ in range(n_portfolios):
    # Random weights from Dirichlet distribution (sums to 1)
    w = np.random.dirichlet(np.ones(n_assets))
    r_, v_, s_ = portfolio_stats(w, ann_returns_vec.values, ann_cov_mat.values)
    rand_rets.append(r_)
    rand_vols.append(v_)
    rand_sharpes.append(s_)

# --- Efficient Frontier Plot ---
fig = go.Figure()

# Random portfolios (colored by Sharpe ratio)
fig.add_trace(go.Scatter(
    x=rand_vols, y=rand_rets,
    mode='markers', marker=dict(
        size=3, color=rand_sharpes, colorscale='Viridis',
        colorbar=dict(title='Sharpe Ratio'), opacity=0.6,
    ), name='Random Portfolios',
))

# Max Sharpe portfolio
fig.add_trace(go.Scatter(
    x=[vol_s], y=[ret_s], mode='markers',
    marker=dict(size=18, color='red', symbol='star', line=dict(width=1, color='white')),
    name=f'Max Sharpe (SR={sr_s:.2f})',
))

# Min Variance portfolio
fig.add_trace(go.Scatter(
    x=[vol_mv], y=[ret_mv], mode='markers',
    marker=dict(size=18, color='blue', symbol='diamond', line=dict(width=1, color='white')),
    name=f'Min Variance (SR={sr_mv:.2f})',
))

# Individual assets
for i, t in enumerate(faang):
    fig.add_trace(go.Scatter(
        x=[np.sqrt(ann_cov_mat.values[i, i])], y=[ann_returns_vec.values[i]],
        mode='markers+text', text=[t], textposition='top center',
        marker=dict(size=12, color=colors[i], line=dict(width=1, color='black')),
        showlegend=False,
    ))

fig.update_layout(
    title='Efficient Frontier -- FAANG Portfolio Optimization',
    xaxis_title='Annualized Volatility',
    yaxis_title='Annualized Return',
    height=550,
    xaxis=dict(tickformat='.0%'), yaxis=dict(tickformat='.0%'),
)
fig.show()

# --- Print optimal weights ---
print('=' * 55)
print('MAXIMUM SHARPE RATIO PORTFOLIO')
print('=' * 55)
for t, w in zip(faang, w_sharpe):
    bar = '#' * int(w * 40)
    print(f'  {t:6s}: {w:6.1%}  {bar}')
print(f'  {"":6s}  Return: {ret_s:.1%} | Vol: {vol_s:.1%} | Sharpe: {sr_s:.2f}')

print()
print('=' * 55)
print('MINIMUM VARIANCE PORTFOLIO')
print('=' * 55)
for t, w in zip(faang, w_minvol):
    bar = '#' * int(w * 40)
    print(f'  {t:6s}: {w:6.1%}  {bar}')
print(f'  {"":6s}  Return: {ret_mv:.1%} | Vol: {vol_mv:.1%} | Sharpe: {sr_mv:.2f}')

---
## 8. Feature Importance Analysis

Understanding *which features* the models rely on is essential for:
- **Trust**: Does the model use economically sensible signals?
- **Debugging**: Is it overfitting to noise features?
- **Iteration**: Which feature groups are most informative?

---

In [ ]:
# ============================================================
# 8.0  Feature Importance from Tree-Based Models
# ============================================================

# Train a final Random Forest on all training data for importance analysis
rf_final = RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=20,
    random_state=RANDOM_SEED, n_jobs=-1
)
rf_final.fit(X[:train_end], y[:train_end])

importances = pd.Series(rf_final.feature_importances_, index=feature_cols)
top_20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_20.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Feature Importances (Random Forest -- Impurity Decrease)', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# XGBoost importance (if available)
if HAS_XGB:
    xgb_final = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, verbosity=0
    )
    xgb_final.fit(X[:train_end], y[:train_end])
    xgb_imp = pd.Series(xgb_final.feature_importances_, index=feature_cols)
    top_20_xgb = xgb_imp.nlargest(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    top_20_xgb.sort_values().plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
    ax.set_title('Top 20 Feature Importances (XGBoost -- Gain)', fontsize=14)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

print('\nInterpretation:')
print('  - Rolling volatility and recent returns often dominate.')
print('  - Technical indicators provide marginal additional signal.')
print('  - Calendar features typically rank low (expected for liquid large-caps).')

---
## 9. Conclusions & Lessons Learned

---

### Key Findings

1. **Return distributions are non-normal**: All FAANG stocks show fat tails and excess kurtosis. Normal-distribution-based risk models (like parametric VaR) systematically underestimate tail risk.

2. **Volatility clusters**: Rolling volatility reveals clear regimes of calm and turbulence. This supports the use of GARCH-family models and regime-switching models in practice.

3. **Correlations are dynamic**: Rolling correlations with SPY vary substantially over time, often spiking during market stress -- precisely when diversification benefits are needed most.

4. **Prediction is extremely difficult**: Even with 40+ carefully engineered features, directional accuracy barely exceeds 50%. This is consistent with the Efficient Market Hypothesis -- if returns were easily predictable, arbitrageurs would trade away the signal.

5. **Walk-forward validation is essential**: Random train/test splits on time-series data produce misleadingly optimistic results due to information leakage.

6. **Diversification works**: The efficient frontier analysis demonstrates meaningful volatility reduction from combining imperfectly correlated assets, even within a single sector (tech).

7. **Monte Carlo illuminates the distribution of outcomes**: Rather than a single point forecast, simulation reveals the full range of possibilities and their probabilities.

### Common Mistakes to Avoid

| Mistake | Why It Invalidates Results |
|---------|---------------------------|
| **Random train/test split** | Leaks future information into training data |
| **Ignoring transaction costs** | Strategies profitable on paper may lose money net of costs |
| **Survivorship bias** | We only analyzed stocks that survived 5 years; failed companies are excluded |
| **Overfitting to backtests** | Optimizing strategy parameters on historical data overfits to that specific period |
| **Ignoring regime changes** | Markets during COVID, 2022 rate hikes, etc. behave fundamentally differently |
| **Confusing correlation with causation** | Features may correlate with returns spuriously |
| **Using future data in features** | Even subtle look-ahead (e.g., using same-day volume to predict same-day close) invalidates everything |
| **Not adjusting for multiple comparisons** | Testing many strategies increases the chance of finding a false positive |

### Extensions Worth Exploring

- **GARCH models** for volatility forecasting and better VaR estimation
- **Transformer architectures** for sequential modeling (temporal attention)
- **Alternative data** (news sentiment, options flow, macro indicators)
- **Ensemble methods** combining predictions from multiple model families
- **Production backtesting frameworks** (e.g., `zipline`, `backtrader`) with realistic transaction costs, slippage, and fill simulation
- **Cross-asset signals** (bonds, commodities, FX data to predict equity returns)
- **Bayesian approaches** for proper uncertainty quantification

---

> **Final reminder**: This is an educational exercise demonstrating quantitative analysis techniques. Real quantitative trading requires years of domain expertise, robust infrastructure, rigorous statistical testing (including corrections for multiple comparisons), out-of-sample validation across market regimes, and -- most importantly -- a deep appreciation for how much we do *not* know about financial markets. Treat any backtest result with skepticism, especially your own.

---

*Built with Python, pandas, scikit-learn, XGBoost, LightGBM, PyTorch/Keras, Plotly, and scipy.*

## Portfolio Quality Addendum

### Objective
Forecast price direction and volatility regimes to support risk-aware trading decisions.

### Data
Daily OHLCV market series with engineered technical indicators and lagged returns.

### Method
Blend trend, momentum, and volatility features with supervised models under time-aware validation.

### Evaluation
Track RMSE/MAE for regression and directional hit-rate for trading relevance.

### Insight and Trade-off
- Insight: Volatility regime shifts dominate error spikes and model instability.
- Because return distributions are non-stationary, static assumptions degrade quickly.
- Therefore regime-aware features and rolling retraining should be the default path.
- Trade-off: richer feature sets can improve edge but raise overfitting risk.
- Limitation: macro-event shocks are underrepresented in historical windows.

## Conclusion and Next Steps

### Summary
The analysis identifies where signal is stable and where market regime changes require guardrails.

### Next Steps
1. Add walk-forward backtesting with transaction-cost assumptions.
2. Benchmark tree models against sequence models under identical splits.
3. Publish a risk dashboard with drawdown and turnover constraints.